In [3]:
# load the dataset cora
import torch
from torch_geometric.data import Data
from dataprocessing.partitioning import label_dirichlet_partition
from dataprocessing.datasets import GraphDataset
from dataprocessing.loaders import load_dataset, load_and_split, load_and_split_with_khop, load_and_split_with_feature_prop


In [14]:
def print_subgraph_stats(data: Data, name: str = "Subgraph"):
    """
    Prints basic statistics for a PyTorch Geometric Data object.
    
    Args:
        data: PyTorch Geometric Data object
        name: Name/identifier for the subgraph (default="Subgraph")
    """
    print(f"{name} stats:")
    print(f"Number of nodes: {data.num_nodes}")
    print(f"Number of edges: {data.edge_index.size(1)}")
    print(f"Number of features: {data.x.size(1)}")
    print(f"Number of classes: {len(torch.unique(data.y))}")
    print(f"Number of training nodes: {data.train_mask.sum().item()}")
    print(f"Number of validation nodes: {data.val_mask.sum().item()}")
    print(f"Number of test nodes: {data.test_mask.sum().item()}")
    print(f"Zero feature vectors: {(data.x.sum(dim=1) == 0).sum().item()}")
    print("-------------------")

# Example usage:
# print_subgraph_stats(subgraph, "Original subgraph")
# print_subgraph_stats(expanded_subgraph, "Expanded subgraph")

In [4]:
data, dataset, clients_data, test_data, client_data_dict = load_and_split_with_khop("Cora", num_clients=10, beta=0.5, hop=1)
client_data_dict

{'Initial number of nodes': 103,
 'K-hop value': 1,
 'Number of nodes after subgraph expansion': 344,
 'Zero feature vectors before FP': 241,
 'Non-zero feature vectors before FP': 103,
 'Percentage of zero vectors before FP': '70.06%',
 'Number of nodes after feature propagation': 344,
 'Zero feature vectors': 241,
 'Non-zero feature vectors': 103,
 'Percentage of zero vectors': '70.06%'}

In [5]:
data, dataset, clients_data, test_data, client_data_dict = load_and_split_with_khop("Cora", num_clients=10, beta=0.5, hop=2)
client_data_dict

{'Initial number of nodes': 103,
 'K-hop value': 2,
 'Number of nodes after subgraph expansion': 1137,
 'Zero feature vectors before FP': 1034,
 'Non-zero feature vectors before FP': 103,
 'Percentage of zero vectors before FP': '90.94%',
 'Number of nodes after feature propagation': 1137,
 'Zero feature vectors': 1034,
 'Non-zero feature vectors': 103,
 'Percentage of zero vectors': '90.94%'}

In [12]:
data, dataset, clients_data, test_data, client_data_dict, split_data_indexes = load_and_split("Cora", num_clients=10, beta=0.5)

In [13]:
data

Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])

In [14]:
len(split_data_indexes)
split1 = split_data_indexes[0]
len(split1)

103

In [15]:
data.x

tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]], device='cuda:0')

In [16]:
print(data.x.shape)
print("First 10 edges:")
for i in range(10):
    source = data.edge_index[0][i].item()
    target = data.edge_index[1][i].item()
    print(f"Edge {i}: {source} → {target}")

torch.Size([2708, 1433])
First 10 edges:
Edge 0: 633 → 0
Edge 1: 1862 → 0
Edge 2: 2582 → 0
Edge 3: 2 → 1
Edge 4: 652 → 1
Edge 5: 654 → 1
Edge 6: 1 → 2
Edge 7: 332 → 2
Edge 8: 1454 → 2
Edge 9: 1666 → 2


In [17]:
data

Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])

In [7]:
print(f"Client 0:   ")
print(f"Number of nodes: {clients_data[0].num_nodes}")
print(f"Number of edges: {clients_data[0].edge_index.size(1)}")
print(f"Number of features: {clients_data[0].x.size(1)}")
print(f"Number of classes: {len(torch.unique(clients_data[0].y))}")
print(f"Number of training nodes: {clients_data[0].train_mask.sum().item()}")
print(f"Number of validation nodes: {clients_data[0].val_mask.sum().item()}")
print(f"Number of test nodes: {clients_data[0].test_mask.sum().item()}")



Client 0:   
Number of nodes: 1137
Number of edges: 4430
Number of features: 1433
Number of classes: 7
Number of training nodes: 67
Number of validation nodes: 210
Number of test nodes: 420


In [8]:
from torch_geometric.utils import k_hop_subgraph
def get_subgraph_1_hop(subgraph: Data, full_graph: Data, node_indices: torch.Tensor):
    """
    Computes the 1-hop subgraph of nodes specified by indices within the full graph.
    
    Args:
        subgraph: Original subgraph Data object
        full_graph: The complete graph Data object
        node_indices: Tensor of node indices in the full graph to get 1-hop neighbors for
    
    Returns:
        Data object containing the 1-hop subgraph
    """
    # Create boolean mask for the specified nodes
    subgraph_nodes_mask = torch.zeros(full_graph.num_nodes, dtype=torch.bool)
    subgraph_nodes_mask[node_indices] = True
    subgraph_nodes = torch.where(subgraph_nodes_mask)[0]

    # Get the 1-hop subgraph
    subset, edge_index, _, _ = k_hop_subgraph(
        subgraph_nodes, 
        1, 
        full_graph.edge_index, 
        relabel_nodes=True
    )

    # Extract features, labels, and masks from the full graph using the subset
    x = full_graph.x[subset]
    y = full_graph.y[subset]
    train_mask = full_graph.train_mask[subset]
    val_mask = full_graph.val_mask[subset]
    test_mask = full_graph.test_mask[subset]

    new_subgraph = Data(x=x, edge_index=edge_index, y=y, 
                       train_mask=train_mask, val_mask=val_mask, test_mask=test_mask)

    return new_subgraph

In [9]:

data, dataset, clients_data, test_data, client_data_dict, split_data_indexes = load_and_split("Cora", num_clients=10, beta=0.5)

subgraph = clients_data[0]
full_graph = data
node_indices = split_data_indexes[0]
# Example usage (assuming you have your subgraph and full_graph Data objects):

print(f"Subgraph nodes: {subgraph}")
print(f"Full graph nodes: {full_graph}")
new_subgraph = get_subgraph_1_hop(subgraph, full_graph, node_indices)

if new_subgraph is not None:
    print(f"New subgraph nodes: {new_subgraph}")
    # ... further processing of new_subgraph ...

Subgraph nodes: Data(x=[103, 1433], edge_index=[2, 54], y=[103], train_mask=[103], val_mask=[103], test_mask=[103])
Full graph nodes: Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])
New subgraph nodes: Data(x=[344, 1433], edge_index=[2, 1056], y=[344], train_mask=[344], val_mask=[344], test_mask=[344])


In [10]:
def prepare_expanded_subgraph_for_propagation(original_subgraph: Data, expanded_subgraph: Data, original_indices: torch.Tensor):
    """
    Prepares expanded subgraph for feature propagation by:
    - Zeroing features of new nodes (non-original nodes)
    - Setting appropriate masks (only original nodes used for training)
    - Maintaining original features and labels for initial nodes
    """
    # Determine device from original subgraph
    device = original_subgraph.x.device
    
    # Get the mapping of original nodes in the expanded graph
    original_nodes_mask = torch.zeros(expanded_subgraph.num_nodes, dtype=torch.bool, device=device)
    
    # The k_hop_subgraph function returns nodes in order where original nodes come first
    # This is guaranteed by the relabel_nodes=True parameter
    original_nodes_mask[:len(original_indices)] = True
    
    # Print some verification info
    print(f"Original nodes: {len(original_indices)}")
    print(f"Expanded nodes: {expanded_subgraph.num_nodes}")
    print(f"Original nodes in expanded graph: {original_nodes_mask.sum().item()}")
    
    # Create new feature matrix (all zeros initially)
    new_x = torch.zeros_like(expanded_subgraph.x, device=device)
    # Copy original features for original nodes
    new_x[original_nodes_mask] = original_subgraph.x
    
    # Create new masks (only original nodes are used for training)
    new_train_mask = torch.zeros(expanded_subgraph.num_nodes, dtype=torch.bool, device=device)
    new_val_mask = torch.zeros(expanded_subgraph.num_nodes, dtype=torch.bool, device=device)
    new_test_mask = torch.zeros(expanded_subgraph.num_nodes, dtype=torch.bool, device=device)
    
    # Copy original masks for original nodes
    new_train_mask[original_nodes_mask] = original_subgraph.train_mask
    new_val_mask[original_nodes_mask] = original_subgraph.val_mask
    new_test_mask[original_nodes_mask] = original_subgraph.test_mask
    
    # Create new labels (zeros for new nodes)
    new_y = torch.zeros(expanded_subgraph.num_nodes, dtype=expanded_subgraph.y.dtype, device=device)
    new_y[original_nodes_mask] = original_subgraph.y
    
    # Create new Data object
    prepared_subgraph = Data(
        x=new_x,
        edge_index=expanded_subgraph.edge_index.to(device),  # Ensure edge_index is also on correct device
        y=new_y,
        train_mask=new_train_mask,
        val_mask=new_val_mask,
        test_mask=new_test_mask
    )
    
    return prepared_subgraph

In [11]:
data, dataset, clients_data, test_data, client_data_dict, split_data_indexes = load_and_split("Cora", num_clients=10, beta=0.5)

subgraph = clients_data[0]
full_graph = data
node_indices = split_data_indexes[0]
# Example usage (assuming you have your subgraph and full_graph Data objects):

print(f"Subgraph nodes: {subgraph}")
print(f"Full graph nodes: {full_graph}")
new_subgraph = get_subgraph_1_hop(subgraph, full_graph, node_indices)

prepared_subgraph = prepare_expanded_subgraph_for_propagation(subgraph, new_subgraph, node_indices)


Subgraph nodes: Data(x=[103, 1433], edge_index=[2, 54], y=[103], train_mask=[103], val_mask=[103], test_mask=[103])
Full graph nodes: Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])
Original nodes: 103
Expanded nodes: 344
Original nodes in expanded graph: 103


In [12]:
# # Example usage with verification
# expanded_subgraph = get_subgraph_1_hop(subgraph, full_graph, original_indices)
# prepared_subgraph = prepare_expanded_subgraph_for_propagation(subgraph, expanded_subgraph, original_indices)

# Verify
print(f"Original features sum: {subgraph.x.sum()}")
print(f"New features sum: {prepared_subgraph.x.sum()}")  # Should be same as original
print(f"Training nodes: {prepared_subgraph.train_mask.sum()}")  # Should match original

Original features sum: 1972.0
New features sum: 1972.0
Training nodes: 11


In [16]:
# print statistics for original
print_subgraph_stats(subgraph, "Original subgraph")
print_subgraph_stats(prepared_subgraph, "Prepared subgraph")

Original subgraph stats:
Number of nodes: 103
Number of edges: 54
Number of features: 1433
Number of classes: 5
Number of training nodes: 11
Number of validation nodes: 16
Number of test nodes: 34
Zero feature vectors: 0
-------------------
Prepared subgraph stats:
Number of nodes: 344
Number of edges: 1056
Number of features: 1433
Number of classes: 5
Number of training nodes: 11
Number of validation nodes: 16
Number of test nodes: 34
Zero feature vectors: 241
-------------------


### Test the functions 

In [17]:
data, dataset, clients_data, test_data, client_data_dict, split_data_indexes = load_and_split("Cora", num_clients=10, beta=0.5)
subgraph = clients_data[0]

# print statistics for original
print_subgraph_stats(subgraph, "Loand and split")


Loand and split stats:
Number of nodes: 103
Number of edges: 54
Number of features: 1433
Number of classes: 5
Number of training nodes: 11
Number of validation nodes: 16
Number of test nodes: 34
Zero feature vectors: 0
-------------------


In [18]:
# test load and split with khop
data, dataset, clients_data, test_data, client_data_dict = load_and_split_with_khop("Cora", num_clients=10, beta=0.5, hop=1)
client_data_dict

print_subgraph_stats(clients_data[0], "Load and split with khop")

Load and split with khop stats:
Number of nodes: 344
Number of edges: 1056
Number of features: 1433
Number of classes: 7
Number of training nodes: 23
Number of validation nodes: 67
Number of test nodes: 120
Zero feature vectors: 241
-------------------


In [20]:
data, dataset, clients_data, test_data, client_data_dict = load_and_split_with_feature_prop("Cora", num_clients=10, beta=0.5, hop=1)
client_data_dict

print_subgraph_stats(clients_data[0], "Load and split with feature prop")


ValueError: too many values to unpack (expected 3)

### Old code


In [27]:
import pytest
import torch
from torch_geometric.data import Data
from dataprocessing.datasets import GraphDataset
from dataprocessing.loaders import (
    load_dataset,
    load_and_split,
    load_and_split_with_khop,
    load_and_split_with_feature_prop
)


In [ ]:
# Regime 1: Basic dataset loading
data, dataset = load_dataset("Cora")
print("Basic Dataset Info:")
print(f"Number of nodes: {data.num_nodes}")
print(f"Number of edges: {data.edge_index.size(1)}")
print(f"Feature dimensions: {data.x.size()}")
print(f"Number of classes: {len(torch.unique(data.y))}")


In [ ]:
# Regime 2: Split into subgraphs
num_clients = 5
beta = 0.5
data, dataset, clients_data, test_data = load_and_split("Cora", num_clients=num_clients, beta=beta)
print("\nRegime 2 - Client Subgraphs:")
for i, client_data in enumerate(clients_data[:2]):  # Show first 2 clients
    print(f"\nClient {i}:")
    print(f"Nodes: {client_data.num_nodes}")
    print(f"Edges: {client_data.edge_index.size(1)}")


In [ ]:
# Regime 3: K-hop neighbors with zero features
hop = 1
data, dataset, clients_khop, test_data = load_and_split_with_khop(
    "Cora", 
    num_clients=num_clients, 
    beta=beta, 
    hop=hop
)
print("\nRegime 3 - K-hop Subgraphs:")
for i, client_data in enumerate(clients_khop[:2]):
    print(f"\nClient {i}:")
    print(f"Nodes: {client_data.num_nodes}")
    print(f"Edges: {client_data.edge_index.size(1)}")
    print(f"Zero feature vectors: {torch.sum(torch.all(client_data.x == 0, dim=1))}")


In [ ]:
# Regime 4: K-hop with feature propagation
data, dataset, clients_prop, test_data = load_and_split_with_feature_prop(
    "Cora", 
    num_clients=num_clients, 
    beta=beta, 
    hop=hop
)
print("\nRegime 4 - Feature Propagation:")
for i, client_data in enumerate(clients_prop[:2]):
    print(f"\nClient {i}:")
    print(f"Nodes: {client_data.num_nodes}")
    print(f"Edges: {client_data.edge_index.size(1)}")
    print(f"Zero feature vectors: {torch.sum(torch.all(client_data.x == 0, dim=1))}")


In [ ]:
# Compare node features across regimes
client_idx = 0  # Pick first client
print("\nFeature Comparison for Client 0:")
print(f"Original subgraph nodes: {clients_data[client_idx].num_nodes}")
print(f"K-hop subgraph nodes: {clients_khop[client_idx].num_nodes}")
print(f"Propagated features nodes: {clients_prop[client_idx].num_nodes}")

In [ ]:
# Regime 1: Basic dataset loading
print("=== Regime 1: Basic Dataset ===")
data, dataset = load_dataset("Cora")
print("Total nodes:", data.num_nodes)
print("Training nodes:", data.train_mask.sum().item())
print("Validation nodes:", data.val_mask.sum().item())
print("Test nodes:", data.test_mask.sum().item())
print("\n")

# Regime 2: Split into subgraphs
print("=== Regime 2: Split Subgraphs ===")
num_clients = 5
beta = 0.5
data, dataset, clients_data, test_data = load_and_split("Cora", num_clients=num_clients, beta=beta)
for i, client_data in enumerate(clients_data):
    print(f"Client {i}:")
    print(f"Total nodes: {client_data.num_nodes}")
    print(f"Training nodes: {client_data.train_mask.sum().item()}")
    print(f"Validation nodes: {client_data.val_mask.sum().item()}")
    print(f"Test nodes: {client_data.test_mask.sum().item()}")
print("\n")

# Regime 3: K-hop neighbors
print("=== Regime 3: K-hop Subgraphs ===")
hop = 1
data, dataset, clients_khop, test_data = load_and_split_with_khop(
    "Cora", 
    num_clients=num_clients, 
    beta=beta, 
    hop=hop
)
for i, client_data in enumerate(clients_khop):
    print(f"Client {i}:")
    print(f"Total nodes: {client_data.num_nodes}")
    print(f"Training nodes: {client_data.train_mask.sum().item()}")
    print(f"Validation nodes: {client_data.val_mask.sum().item()}")
    print(f"Test nodes: {client_data.test_mask.sum().item()}")
print("\n")

# Regime 4: K-hop with feature propagation
print("=== Regime 4: Feature Propagation ===")
data, dataset, clients_prop, test_data = load_and_split_with_feature_prop(
    "Cora", 
    num_clients=num_clients, 
    beta=beta, 
    hop=hop
)
for i, client_data in enumerate(clients_prop):
    print(f"Client {i}:")
    print(f"Total nodes: {client_data.num_nodes}")
    print(f"Training nodes: {client_data.train_mask.sum().item()}")
    print(f"Validation nodes: {client_data.val_mask.sum().item()}")
    print(f"Test nodes: {client_data.test_mask.sum().item()}")

# Summary statistics
print("\n=== Summary Statistics ===")
print("Original dataset total nodes:", data.num_nodes)
print("\nAverage nodes per client:")
print(f"Regime 2: {sum(c.num_nodes for c in clients_data) / num_clients:.2f}")
print(f"Regime 3: {sum(c.num_nodes for c in clients_khop) / num_clients:.2f}")
print(f"Regime 4: {sum(c.num_nodes for c in clients_prop) / num_clients:.2f}")

print("\nAverage training nodes per client:")
print(f"Regime 2: {sum(c.train_mask.sum().item() for c in clients_data) / num_clients:.2f}")
print(f"Regime 3: {sum(c.train_mask.sum().item() for c in clients_khop) / num_clients:.2f}")
print(f"Regime 4: {sum(c.train_mask.sum().item() for c in clients_prop) / num_clients:.2f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Load data in different regimes
num_clients = 5
beta = 0.5
hop = 1

# Get data with zero features (Regime 3)
_, _, clients_khop, _ = load_and_split_with_khop(
    "Cora", 
    num_clients=num_clients, 
    beta=beta, 
    hop=hop
)

# Get data with propagated features (Regime 4)
_, _, clients_prop, _ = load_and_split_with_feature_prop(
    "Cora", 
    num_clients=num_clients, 
    beta=beta, 
    hop=hop
)

# Select a client and a few features to visualize
client_idx = 0  # First client
feature_idx = [0, 1, 2]  # First three features

# Get features
zero_features = clients_khop[client_idx].x
prop_features = clients_prop[client_idx].x

# Create visualization
plt.figure(figsize=(15, 5))

# Plot 1: Feature distribution before propagation
plt.subplot(121)
for i, feat in enumerate(feature_idx):
    sns.kdeplot(zero_features[:, feat].cpu().numpy(), label=f'Feature {feat}')
plt.title('Feature Distribution Before Propagation')
plt.xlabel('Feature Value')
plt.ylabel('Density')
plt.legend()

# Plot 2: Feature distribution after propagation
plt.subplot(122)
for i, feat in enumerate(feature_idx):
    sns.kdeplot(prop_features[:, feat].cpu().numpy(), label=f'Feature {feat}')
plt.title('Feature Distribution After Propagation')
plt.xlabel('Feature Value')
plt.ylabel('Density')
plt.legend()

plt.tight_layout()
plt.show()

# Heatmap of a few nodes before and after propagation
n_nodes = 10  # Number of nodes to visualize
n_features = 10  # Number of features to visualize

plt.figure(figsize=(15, 5))

# Plot 1: Features before propagation
plt.subplot(121)
sns.heatmap(zero_features[:n_nodes, :n_features].cpu(), 
            cmap='viridis', 
            xticklabels=range(n_features),
            yticklabels=range(n_nodes))
plt.title('Node Features Before Propagation')
plt.xlabel('Feature Index')
plt.ylabel('Node Index')

# Plot 2: Features after propagation
plt.subplot(122)
sns.heatmap(prop_features[:n_nodes, :n_features].cpu(), 
            cmap='viridis',
            xticklabels=range(n_features),
            yticklabels=range(n_nodes))
plt.title('Node Features After Propagation')
plt.xlabel('Feature Index')
plt.ylabel('Node Index')

plt.tight_layout()
plt.show()

# Print some statistics
print("\nFeature Statistics:")
print("\nBefore Propagation:")
print(f"Zero vectors: {torch.sum(torch.all(zero_features == 0, dim=1)).item()}")
print(f"Mean value: {zero_features.mean().item():.4f}")
print(f"Std value: {zero_features.std().item():.4f}")

print("\nAfter Propagation:")
print(f"Zero vectors: {torch.sum(torch.all(prop_features == 0, dim=1)).item()}")
print(f"Mean value: {prop_features.mean().item():.4f}")
print(f"Std value: {prop_features.std().item():.4f}")

In [22]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from dataprocessing.loaders import load_dataset
from dataprocessing.utils import get_propagation_matrix

def propagate_features_with_monitoring(x, edge_index, mask, max_iterations=50, tolerance=1e-6):
    """
    Propagate features and monitor convergence
    
    Args:
        x: Node features
        edge_index: Graph connectivity
        mask: Boolean mask for known features
        max_iterations: Maximum number of iterations
        tolerance: Convergence threshold
    """
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    x = x.to(DEVICE)
    mask = mask.bool().to(DEVICE)
    edge_index = edge_index.to(DEVICE)
    
    # Initialize
    if mask is not None:
        out = torch.zeros_like(x)
        out[mask] = x[mask]
    else:
        out = x.clone()
    
    n_nodes = x.size(0)
    adj = get_propagation_matrix(out, edge_index, n_nodes)
    
    # Track changes
    changes = []
    no_of_zero_vectors = []
    
    for iter in range(max_iterations):
        # Store previous state
        prev_out = out.clone()
        no_of_zero_vectors.append(torch.sum(torch.all(out == 0, dim=1)).item())
        
        # Propagate
        out = torch.sparse.mm(adj, out)
        out[mask] = x[mask]  # Reset known features
        
        # Calculate change
        change = torch.norm(out - prev_out).item()
        changes.append(change)
        
        # Check convergence
        # if change < tolerance:
        #     print(f"Converged at iteration {iter}")
        #     break
            
    return out, changes, no_of_zero_vectors


In [ ]:

# Load data and test convergence
data, _ = load_dataset("Cora")
num_clients = 5
beta = 0.5
hop = 1

# Get a client's data
_, _, clients_prop, _ = load_and_split_with_feature_prop(
    "Cora", 
    num_clients=num_clients, 
    beta=beta, 
    hop=hop
)

client_data = clients_prop[0]  # First client
zero_vector_mask = (client_data.x == 0).all(dim=1)
non_zero_vector_mask = ~zero_vector_mask

# Propagate features with monitoring
propagated_features, changes, no_of_zero_vectors = propagate_features_with_monitoring(
    client_data.x,
    client_data.edge_index,
    non_zero_vector_mask
)

# Plot convergence
plt.figure(figsize=(10, 5))
plt.plot(changes)
plt.yscale('log')
plt.xlabel('Iteration')
plt.ylabel('Change in Features (L2 norm)')
plt.title('Feature Propagation Convergence')
plt.grid(True)
plt.show()

# Print statistics
print(f"\nFinal change: {changes[-1]:.8f}")
print(f"Number of iterations: {len(changes)}")